In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#root="/content/drive/MyDrive/pr"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pickle
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
train_dir = '/content/drive/MyDrive/pr/train'

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_files = []
train_labels = []

for class_label in [0, 1]:
    class_dir = os.path.join(train_dir, f'class_{class_label}')
    for f in os.listdir(class_dir):
        if f.endswith('.pkl'):
            train_files.append(os.path.join(class_dir, f))
            train_labels.append(class_label)

train_files, val_files, train_labels, val_labels = train_test_split(train_files, train_labels, test_size=0.1, random_state=42)  #split the train_data into (len(train_data), len(val_data)) = 9:1

print(len(train_files))
print(len(val_files))


270
30


In [ ]:
# transform the image into the type that can be more fit to resnet18
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
class train_BagDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]

        with open(file_path, 'rb') as f:
            images = pickle.load(f)

        if self.transform:
            images = [self.transform(image) for image in images]
        images = torch.stack(images)

        return images, label

In [ ]:

class BagNet(nn.Module):
    def __init__(self):
        super(BagNet, self).__init__()
        self.resnet18 = models.resnet18(pretrained=True)
        self.resnet18 = nn.Sequential(*list(self.resnet18.children())[:-1])  #remove the last layer to fit the given data
        self.dropout = nn.Dropout(p=0.5)  #use droppout to prevent the overfitting
        self.fc1 = nn.Linear(512, 128)
        self.fc2 = nn.Linear(128, 1) #it is one because it need to fit the BCEloss

    def forward(self, x):
        batch_size, num_images, C, H, W = x.shape
        x = x.view(batch_size * num_images, C, H, W)
        features = self.resnet18(x)  #extract the feature
        features = features.view(batch_size, num_images, -1)
        bag_features = torch.mean(features, dim=1)  #to get the mean of the bag
        bag_features = self.dropout(bag_features)
        x = self.fc1(bag_features)
        x = torch.relu(x)
        x = self.fc2(x)
        return x.squeeze(1)


In [ ]:
train_dataset = train_BagDataset(train_files, train_labels, transform=transform)  #load the train_data into the dataset class
val_dataset = train_BagDataset(val_files, val_labels, transform=transform)  #load the val_data into the dataset class

train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=6, shuffle=False)

In [ ]:
model = BagNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
def plot_learning_curve(train_losses, val_losses, val_accuracies):

    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 4))
    # the loss part
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'bo-', label='Training loss')
    plt.plot(epochs, val_losses, 'ro-', label='Validation loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    #plt.show()

    #the accuracy part
    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_accuracies, 'go-', label='Validation accuracy')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

In [ ]:

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, save_dir='model_weights'):

    os.makedirs(save_dir, exist_ok=True)

    Train_loss = []
    Val_loss = []
    Val_accuracy = []

    for epoch in range(num_epochs):

        #training part
        model.train()
        running_loss = 0.0
        trainloader = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device).float()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        Train_loss.append(train_loss)
        trainloader.set_postfix(loss=train_loss)

        # validation part
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", unit="batch"):
                images, labels = images.to(device), labels.to(device).float()
                outputs = model(images)
                #print(outputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                predicted = torch.sigmoid(outputs) > 0.5
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        Val_loss.append(val_loss)
        val_accuracy = correct / total
        Val_accuracy.append(val_accuracy)
        print("val_loss: ", val_loss/len(val_loader))
        print("Accuracy: ", val_accuracy)

        model.train()
        save_path = os.path.join(save_dir, f'train_epoch_{epoch+1}.pth')
        torch.save(model.state_dict(), save_path)

    plot_learning_curve(Train_loss, Val_loss, Val_accuracy)

train_model(model, train_loader, val_loader, criterion, optimizer)